In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

2026-03-10 15:53:49.771204: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-10 15:53:49.775190: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/system/software/code-server/4.107.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cU

In [2]:
import pandas as pd
import numpy as np

In [3]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=3:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=4)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

2026-03-10 15:54:26,137 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 12.00 GiB
2026-03-10 15:54:26,138 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 12.00 GiB


In [4]:
client.dashboard_link

'http://127.0.0.1:8787/status'

In [5]:
DATA_ROOT="/nfs/roberts/project/pi_skr2/shared/tabula_data"
pkl_path=f"{DATA_ROOT}/shendure/shendure_ortho_20260306/training_data.pkl"
import pickle as pkl
with open(pkl_path,"rb") as f:
    dat=pkl.load(f)

In [6]:
dat.set_consider_missing(True)

In [7]:
dat.data

,rep_id,mpra_bc,cre_id,umis_transfection_bc,transfection_bc,reads_mpra_bc,umis_mpra_bc,cell_type,cell_bc,reads_transfection_bc,cre_id_original
npartitions=1,,,,,,,,,,,
,string,string,string,int64,string,"Sparse[int64, 0]",int64,string,string,"Sparse[int64, 0]",string
,...,...,...,...,...,...,...,...,...,...,...


manually calc reference beta...

In [ ]:
client.close()
cluster.close()